# Conditionally clean system qubits in int2float's XAG circuit

This experiment uses the saved **`int2float_xag_circuit.svg`**, which represents
**output 3**, not all seven outputs of int2float. Its companion QPY/JSON provides
the exact circuit: 11 inputs (`q0..q10`), output `q11`, seven scratch qubits
(`q12..q18`), and 32 operations. The SVG hides idle `q0` and `q1`; all resource
comparisons here retain the same 11-input interface.

**Result:** borrowing q9 and q4 reduces allocated width **19 → 17**.
The additional workspace reuse proposed by the user, **C into q12**, combines
with those moves to reach **19 → 16**, with five 3-controlled X gates.
A separate clean-reuse alternative saves q14 using only CCX gates, at the cost
of two additional CCX operations. These are explicit verified experiments,
not a general synthesis optimizer or a proof of optimality.

We first adopt the requested **CCX Boolean abstraction**. The export contains
CCRx(+π) in computation and CCRx(−π) in cleanup, so both signs become CCX for this
analysis. These are not gate-by-gate unitary identities: Rx(+π)=−iX and Rx(−π)=iX.
Later we compare the complete transformed circuits against the original rotations,
including their phases, on the clean-scratch subspace.

Run locally, without native synthesis or network access:
```bash
uv run --no-sync logique notebook run experiments/studies/int2float_conditional_workspace.ipynb --offline
```
The source stays output-free; executed copies, tables, QPY circuits, and figures
are saved under a fresh `dump/` directory.

In [ ]:
from collections import Counter
from copy import deepcopy
from importlib.metadata import version
import hashlib
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from qiskit import QuantumCircuit, QuantumRegister, qpy
from qiskit.quantum_info import Operator, Statevector
from logique.benchmarks.runner import new_run_directory
from logique.circuits.records import records_to_qiskit, _qiskit_records
from logique.verification.classical import (
    _apply_record,
    input_assignments,
    truth_outputs,
)

ROOT = Path(os.environ.get("LOGIQUE_WORKSPACE", Path.cwd())).resolve()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SOURCE = ROOT / "datasets/boolean/studies/int2float_xag_output3.json"
DATA = json.loads(SOURCE.read_text())
OUT = (
    Path(os.environ["LOGIQUE_OUTPUT"])
    if "LOGIQUE_OUTPUT" in os.environ
    else new_run_directory()
)
ATOL = 1e-10
INPUTS = input_assignments(11)
ORIGINAL = DATA["gates"]
REFERENCE = []
for record in ORIGINAL:
    if record["kind"] == "rx":
        assert len(record["controls"]) == 2
        assert np.isclose(abs(record["angle"]), np.pi)
    else:
        assert record["kind"] == "x"
    REFERENCE.append({k: v for k, v in record.items() if k != "angle"} | {"kind": "x"})
print("Input snapshot SHA256:", hashlib.sha256(SOURCE.read_bytes()).hexdigest())
print("Benchmark output:", DATA["selected_outputs"], "— target q11")
print({name: version(name) for name in ("qiskit", "numpy", "nbformat")})
print("Results:", OUT)

## 1. Boolean expression and the predicate that matters

Let $x_i$ denote the **original** value of input `qi`. The forward computation is

$$A=x_2x_8,\quad B=\overline{x_8\oplus x_7}\,\overline A,\quad
C=\overline{x_8\oplus x_5}\,B,$$
$$D=\overline{x_4\oplus x_8}\,\overline{x_8\oplus x_6},\quad
E=CD,\quad F=\overline{x_9}E,\quad G=\overline{x_{10}}F,$$
$$f=\overline{\overline{x_3}G}.$$

The last X on `q11` implements this outer complement. It must not be omitted.
The source logic network supplies a separate specification: the notebook checks
the expressions above against that network on all 2,048 inputs.

Unlike our previous toy circuit, the final consumer does **not** include A.
In fact, B, C, and E require $A=0$. Borrowing `q2` merely because
$A=1\Rightarrow x_2=1$ therefore protects the wrong branch.

In [ ]:
A = INPUTS[2] & INPUTS[8]
B = ~(INPUTS[8] ^ INPUTS[7]) & ~A
C = ~(INPUTS[8] ^ INPUTS[5]) & B
D = ~(INPUTS[4] ^ INPUTS[8]) & ~(INPUTS[8] ^ INPUTS[6])
E = C & D
F = ~INPUTS[9] & E
G = ~INPUTS[10] & F
EXPECTED = ~(~INPUTS[3] & G)
np.testing.assert_array_equal(
    EXPECTED, truth_outputs(DATA["source_network"], INPUTS)[0]
)
assert int(EXPECTED.sum()) == 2036
SYMBOLS = dict(zip("ABCDEFG", (A, B, C, D, E, F, G)))
STEPS = [
    (1, "A", 12),
    (3, "B", 13),
    (6, "C", 14),
    (10, "D", 15),
    (13, "E", 16),
    (14, "F", 17),
    (15, "G", 18),
]


def gate(controls: list[int], target: int, polarity: list[bool] | None = None) -> dict:
    """Describe an exact controlled X using original wire numbers."""
    return {
        "kind": "x",
        "controls": controls,
        "polarity": polarity if polarity is not None else [True] * len(controls),
        "target": target,
    }


def propagate(records: list[dict], logical_bits: int = 11, initial=None):
    """Track every allowed basis column, its exact phase, and intermediate bits."""
    if initial is None:
        bits = np.zeros((19, 1 << logical_bits), dtype=bool)
        bits[:logical_bits] = input_assignments(logical_bits)
    else:
        bits = initial.copy()
    phases = np.ones(bits.shape[1], dtype=complex)
    trace = [bits.copy()]
    for record in records:
        _apply_record(bits, phases, record)
        trace.append(bits.copy())
    return bits, phases, trace


_, _, REFERENCE_TRACE = propagate(REFERENCE)
for after_gate, symbol, wire in STEPS:
    np.testing.assert_array_equal(REFERENCE_TRACE[after_gate][wire], SYMBOLS[symbol])
display(
    pd.DataFrame(
        [
            {
                "gate": i + 1,
                "target": f"q{g['target']}",
                "controls": ", ".join(
                    f"q{q}={int(p)}" for q, p in zip(g["controls"], g["polarity"])
                )
                or "none",
            }
            for i, g in enumerate(REFERENCE)
        ]
    )
)

## 2. Which system wires are conditionally clean?

We inspect **all 11 input wires** immediately after each clean-target AND in the
forward computation. For each stored flag, both its 0 and 1 branches are tested.
A row is accepted only if the branch is nonempty and the wire has one fixed value
on every input in that branch. `valid_before_gate` is the next write to either
the wire or its protecting flag: a conservative boundary, not permission to borrow.

This inventory covers single stored-flag predicates at those checkpoints, not
all possible composite predicates or alternative schedules. Values refer to the
**current wire**. For example, after gate 3, B=1 implies `q8=0`, but this wire
currently holds $x_8\oplus x_7$, not $x_8$. The next CNOT changes that fact.

In [ ]:
facts = []
for after_gate, symbol, flag in STEPS:
    state = REFERENCE_TRACE[after_gate]
    for flag_value in (0, 1):
        branch = state[flag] == flag_value
        if not branch.any():
            continue
        for wire in DATA["input_qubits"]:
            values = state[wire, branch]
            if not np.all(values == values[0]):
                continue
            expiry = next(
                (
                    i + 1
                    for i in range(after_gate, len(REFERENCE))
                    if REFERENCE[i]["target"] in (wire, flag)
                ),
                len(REFERENCE) + 1,
            )
            facts.append(
                {
                    "after_gate": after_gate,
                    "predicate": f"{symbol}={flag_value}",
                    "stored_flag": f"q{flag}",
                    "system_wire": f"q{wire}",
                    "known_value": int(values[0]),
                    "branch_inputs": int(branch.sum()),
                    "valid_before_gate": expiry,
                }
            )
facts_table = pd.DataFrame(facts)
display(facts_table)
facts_table.to_csv(OUT / "conditional_system_facts.csv", index=False)
assert any(row["predicate"] == "F=1" and row["system_wire"] == "q9" for row in facts)
assert any(row["predicate"] == "D=1" and row["system_wire"] == "q4" for row in facts)

The useful facts and their limits are:

| Predicate | Known system values at the stated boundary | Consequence |
|---|---|---|
| A=1, after gate 1 | q2=q8=1 | Normalize with X if borrowed, but A is the wrong guard for the later output logic. |
| B=1, after gate 3 | q8=0 | Transient XOR value; gate 4 changes it. |
| C=1, after gate 6 | q8=0 | Transient XOR value; gate 7 changes it. |
| D=1, after gate 10 | q4=q8=0 | q8 is restored at gate 11; q4 remains suitable if gate 12 is delayed. |
| E=1, after gate 13 | No fixed individual input | It implies correlations, not globally known input values. |
| F=1, after gate 14 | q9=0 | The next Toffoli can target q9, with guarded consumption. |
| G=1, after gate 15 | q9=q10=0 | The only remaining forward target is the externally visible output. |

There are also **composite** conditions. On $E=1$, all of $x_4,x_5,x_6,x_7,x_8$
are equal. Conditioning additionally on q8 exposes more known inputs. The next
cell excludes q8 itself, since it is a guard, and does not claim a free guard or
a verified rewrite for these extra candidates.

In [ ]:
composite_facts = []
state = REFERENCE_TRACE[13]
for value in (0, 1):
    mask = state[16] & (state[8] == value)
    assert mask.any()
    for wire in DATA["input_qubits"]:
        if wire == 8:
            continue
        if np.all(state[wire, mask] == state[wire, mask][0]):
            composite_facts.append(
                {
                    "predicate": f"E=1 AND q8={value}",
                    "system_wire": f"q{wire}",
                    "known_value": int(state[wire, mask][0]),
                    "branch_inputs": int(mask.sum()),
                }
            )
display(pd.DataFrame(composite_facts))

## 3. First relocation: G moves from q18 to q9

After storing $F=\overline{x_9}E$ in q17, the predicate $F=1$ guarantees $x_9=0$.
Redirect the subsequent Toffoli `X(q18) controlled by q10=0,q17=1` to **q9**:

$$r_9=x_9\oplus(\overline{x_{10}}F)=x_9\oplus G,\qquad
F=1\Rightarrow r_9=G.$$

The old output consumer $\overline{x_3}G$ must become
$\overline{x_3}F r_9$. This is a **3-controlled X**, because we retain the flag F.
The identity $F r_9=G$ is global, despite $r_9=G$ holding only conditionally.

Then reverse the modified prefix: restore q9 before erasing F, whose original
computation reads q9. Finish with the original X on q11. Scratch q18 is eliminated.

In [ ]:
def complete_compute(prefix: list[dict], consumer: dict) -> list[dict]:
    """Toggle the output, undo the exact prefix, then apply its outer complement."""
    return deepcopy(prefix) + [consumer] + deepcopy(prefix[::-1]) + [gate([], 11)]


ONE_PREFIX = deepcopy(REFERENCE[:15])
ONE_PREFIX[-1]["target"] = 9
ONE = complete_compute(ONE_PREFIX, gate([3, 17, 9], 11, [False, True, True]))
_, _, one_trace = propagate(ONE_PREFIX)
np.testing.assert_array_equal(one_trace[14][9, F], 0)
np.testing.assert_array_equal(one_trace[15][17], F)
np.testing.assert_array_equal(one_trace[15][9], INPUTS[9] ^ G)
np.testing.assert_array_equal(F & one_trace[15][9], G)
print("PASS: q9 holds G on F=1; the preserved flag makes its consumer exact.")

## 4. Second relocation: E moves from q16 to q4

After gate 10, q4 contains $u=x_4\oplus x_8$, and
$D=\overline u\,\overline{x_8\oplus x_6}$ is stored on q15.
Thus **D=1 guarantees q4=0**. Gate 11 restores q8 without changing this fact.
Instead of executing the q4-restoration CNOT at gate 12, keep q4 in this encoding
until after its borrowed value has been consumed and uncomputed.

Redirect the Toffoli computing E into q4:

$$r_4=u\oplus CD,\qquad D=1\Rightarrow r_4=E.$$

Compute F with the guarded consumer $\overline{x_9}D r_4$ into q17. Since
$D r_4=CD=E$, this still computes the **globally correct F**. Its original two
controls become three. Now perform the first relocation above: G into q9 and
the output consumer guarded by F.

The inverse restores q9, erases F, erases the borrowed E contribution in q4,
uncomputes D, then eventually reverses the original CNOT that produced u.
All inputs are restored. Both old q16 and q18 are eliminated.

In [ ]:
TWO_PREFIX = deepcopy(REFERENCE[:11])  # Stop before restoring q4 at original gate 12.
TWO_PREFIX += [
    gate([14, 15], 4),
    gate([9, 15, 4], 17, [False, True, True]),
    gate([10, 17], 9, [False, True]),
]
TWO = complete_compute(TWO_PREFIX, gate([3, 17, 9], 11, [False, True, True]))
_, _, two_trace = propagate(TWO_PREFIX)
u = INPUTS[4] ^ INPUTS[8]
np.testing.assert_array_equal(two_trace[11][4], u)
np.testing.assert_array_equal(two_trace[11][4, D], 0)
np.testing.assert_array_equal(two_trace[12][4], u ^ E)
np.testing.assert_array_equal(D & two_trace[12][4], E)
np.testing.assert_array_equal(two_trace[13][17], F)
np.testing.assert_array_equal(two_trace[14][9], INPUTS[9] ^ G)
np.testing.assert_array_equal(F & two_trace[14][9], G)
print(
    "PASS: D protects the first borrowed result; the globally correct F protects the next."
)

### 4.1 Store C in q12: borrowing an existing workspace wire

**Yes: q12 is conditionally clean under B=1.** The earlier inventory scanned
only system inputs, so it omitted this existing workspace wire. At this point
q12 holds $A$, and $B=\overline{x_8\oplus x_7}\,\overline A$ implies

$$B=1\quad\Rightarrow\quad q_{12}=A=0.$$

The third Toffoli (gate 6) computes C and is itself controlled by B. Redirecting
its target from q14 to q12 therefore gives

$$r_{12}=A\oplus C,\qquad B=1\Rightarrow r_{12}=C.$$

This is not a global replacement of A by C. On B=0, C=0 and q12 still contains A.
To compute E correctly, change its consumer from $CD$ to **$B r_{12}D$**:

$$B r_{12}=B(A\oplus C)=BA\oplus BC=0\oplus C=C.$$

The consumer becomes a 3-controlled X. Its cleanup inverse must use the same
controls. Reverse the relocated C gate before uncomputing B; this restores A
in q12 before the inverse B computation reads it. No extra X is needed to
normalize q12, because its known value under B is already zero.

This move alone eliminates q14: **19 → 18** qubits. Combined with the earlier
q4 and q9 borrowings, it eliminates q14, q16, and q18: **19 → 16** qubits.
The combined chain preserves the successive predicates B, D, and F.

**A separate, globally clean alternative:** after B has been computed and q8
restored (after gate 4), undo A with the original Toffoli, making q12 zero on
*every* input. Then compute C into q12. Reversing this extended prefix later
recomputes A before B is erased. This also saves q14, uses no extra guard
controls, and costs two additional CCX gates in the complete computation.
That is ordinary uncomputation/recomputation, rather than conditional borrowing.

In [ ]:
Q12_PREFIX = deepcopy(REFERENCE[:15])
Q12_PREFIX[5]["target"] = 12
Q12_PREFIX[12] = gate([13, 12, 15], 16)  # E = B * r12 * D.
Q12 = complete_compute(Q12_PREFIX, REFERENCE[15])
_, _, q12_trace = propagate(Q12_PREFIX)
np.testing.assert_array_equal(q12_trace[5][12, B], 0)
np.testing.assert_array_equal(q12_trace[6][12], A ^ C)
np.testing.assert_array_equal(q12_trace[6][13], B)
np.testing.assert_array_equal(B & q12_trace[6][12], C)
np.testing.assert_array_equal(q12_trace[13][16], E)

THREE_PREFIX = deepcopy(TWO_PREFIX)
THREE_PREFIX[5]["target"] = 12
THREE_PREFIX[11] = gate([13, 12, 15], 4)  # Add B while retaining the D guard.
THREE = complete_compute(THREE_PREFIX, gate([3, 17, 9], 11, [False, True, True]))
_, _, three_trace = propagate(THREE_PREFIX)
np.testing.assert_array_equal(three_trace[6][12], A ^ C)
np.testing.assert_array_equal(three_trace[12][4], u ^ E)
np.testing.assert_array_equal(D & three_trace[12][4], E)
np.testing.assert_array_equal(three_trace[13][17], F)
np.testing.assert_array_equal(F & three_trace[14][9], G)

CLEAN_REUSE_PREFIX = deepcopy(REFERENCE[:15])
CLEAN_REUSE_PREFIX[5]["target"] = 12
CLEAN_REUSE_PREFIX[12]["controls"] = [12, 15]
CLEAN_REUSE_PREFIX.insert(4, gate([2, 8], 12))  # Uncompute A after restoring q8.
CLEAN_REUSE = complete_compute(CLEAN_REUSE_PREFIX, REFERENCE[15])
_, _, clean_reuse_trace = propagate(CLEAN_REUSE_PREFIX)
assert not clean_reuse_trace[5][12].any()
np.testing.assert_array_equal(clean_reuse_trace[7][12], C)
print(
    "PASS: conditional q12 reuse, combined B/D/F borrowing, and globally clean reuse."
)

## 5. Failed attempts are part of the experiment

1. Moving G into q9 **without adding F to the consumer** is wrong when F=0 and
   the original x9=1. Even if cleanup restores everything, it computes the wrong
   output and therefore the wrong phase oracle.
2. Moving C into flipped q2 based on **A=1**, then using that wire as C without
   a valid guard, fails because the desired computation instead needs A=0.
3. Borrowing q4 while leaving the original early-restoration CNOT in place uses
   q4 after its known-zero condition has expired.

4. Moving C into q12 **without retaining B in E’s consumer** incorrectly
   consumes the residual A on B=0.

Each rejected circuit below still has an exact inverse cleanup. Restoration
alone is therefore insufficient. Counterexamples are reported with bits ordered
`x0 x1 ... x10`, not Qiskit's reversed display order.

In [ ]:
NO_GUARD = complete_compute(ONE_PREFIX, gate([3, 9], 11, [False, True]))
wrong_branch_prefix = deepcopy(REFERENCE[:15])
wrong_branch_prefix[5]["target"] = 2
wrong_branch_prefix[12]["controls"] = [2, 15]
wrong_branch_prefix.insert(5, gate([], 2))
WRONG_BRANCH = complete_compute(wrong_branch_prefix, REFERENCE[15])
early_restore_prefix = deepcopy(REFERENCE[:12])
early_restore_prefix += deepcopy(TWO_PREFIX[11:])
EARLY_RESTORE = complete_compute(
    early_restore_prefix, gate([3, 17, 9], 11, [False, True, True])
)

unguarded_q12_prefix = deepcopy(Q12_PREFIX)
unguarded_q12_prefix[12] = gate([12, 15], 16)
UNGUARDED_Q12 = complete_compute(unguarded_q12_prefix, REFERENCE[15])

failures = []
for label, records in [
    ("Missing F guard", NO_GUARD),
    ("Wrong A branch", WRONG_BRANCH),
    ("q4 restored too early", EARLY_RESTORE),
    ("q12 borrowed without B guard", UNGUARDED_Q12),
]:
    bits, phases, _ = propagate(records)
    np.testing.assert_array_equal(bits[:11], INPUTS)
    assert not bits[12:].any()
    np.testing.assert_allclose(phases, 1, atol=ATOL, rtol=0)
    mismatches = np.flatnonzero(bits[11] != EXPECTED)
    assert mismatches.size > 0
    example = int(mismatches[0])
    failures.append(
        {
            "attempt": label,
            "wrong_inputs": int(mismatches.size),
            "first_x0_to_x10": "".join(str(int(v)) for v in INPUTS[:, example]),
            "expected_f": int(EXPECTED[example]),
            "actual_f": int(bits[11, example]),
        }
    )
assert failures[0]["wrong_inputs"] == 512
assert failures[1]["wrong_inputs"] == 32
assert failures[3]["wrong_inputs"] == 16
display(pd.DataFrame(failures))

## 6. Exhaustive output, phase, and restoration verification

All gates here are **monomial**: each basis vector maps to one basis vector with
a phase. Track both exactly instead of constructing a dense 19-qubit unitary.
We check all **4,096 input-and-target basis columns**, with arbitrary initial q11
and zero scratch, against the independent network specification. Fixed complex
phases are compared directly, not up to a different global phase for each input.
This proves the XOR action on arbitrary superpositions in that subspace.

We separately check all **2,048 clean-target basis columns** through
compute–Z(q11)–uncompute. Inputs, target, and scratch must return to their initial
values, with exactly $(-1)^{f(x)}$. The original ±π rotations are included as a
baseline, so the endpoint comparison goes beyond the CCX abstraction. No claim
is made that independently replacing a rotation by X is a unitary identity.

In [ ]:
VARIANTS = {
    "original_rotations": ORIGINAL,
    "ccx_reference": REFERENCE,
    "borrow_q9": ONE,
    "borrow_q4_q9": TWO,
    "borrow_q12": Q12,
    "borrow_q12_q4_q9": THREE,
    "clean_reuse_q12": CLEAN_REUSE,
}
verification = []
for name, records in VARIANTS.items():
    bits, phases, _ = propagate(records, logical_bits=12)
    logical = input_assignments(12)
    np.testing.assert_array_equal(bits[:11], logical[:11])
    np.testing.assert_array_equal(bits[11], logical[11] ^ np.tile(EXPECTED, 2))
    assert not bits[12:].any()
    np.testing.assert_allclose(phases, 1, atol=ATOL, rtol=0)

    phase_bits, phase_factors, _ = propagate(records)
    phase_factors *= 1 - 2 * phase_bits[11].astype(int)
    for record in reversed(records):
        _apply_record(phase_bits, phase_factors, record, inverse=True)
    np.testing.assert_array_equal(phase_bits[:11], INPUTS)
    assert not phase_bits[11:].any()
    np.testing.assert_allclose(
        phase_factors, 1 - 2 * EXPECTED.astype(int), atol=ATOL, rtol=0
    )
    verification.append(
        {
            "circuit": name,
            "xor_basis_columns": 4096,
            "phase_basis_columns": 2048,
            "wrong_outputs": 0,
            "input_and_scratch_restored": True,
            "exact_phase": True,
        }
    )
display(pd.DataFrame(verification))

## 7. Allocate only the wires that remain

Keep the 11 input wires and output q11, then compact the retained scratch wires.
For the two-borrow construction, old q17 becomes physical q16. Diagrams use the
**original labels** to keep the gate relocation visible. The mapping table and
QPY metadata record the distinction.

Removing idle input wires q0 and q1 would save two wires in *every* circuit; that
is independent of the conditional-workspace saving and is not included here.

In [ ]:
def compact_circuit(records: list[dict], name: str):
    """Keep the input/output interface and physically remove unused scratch."""
    retained = sorted(
        set(range(12)) | {q for g in records for q in [*g["controls"], g["target"]]}
    )
    mapping = {old: new for new, old in enumerate(retained)}
    compacted = deepcopy(records)
    for record in compacted:
        record["controls"] = [mapping[q] for q in record["controls"]]
        record["target"] = mapping[record["target"]]
    circuit = records_to_qiskit(compacted, len(retained))
    circuit.name = name
    circuit.metadata = {
        "original_to_physical": {str(q): v for q, v in mapping.items()},
        "selected_output": 3,
        "output_qubit": mapping[11],
    }
    # Re-read the actual Qiskit export to validate its gates, polarity, and wiring.
    exported = _qiskit_records(circuit)
    physical_initial = np.zeros((19, 4096), dtype=bool)
    physical_initial[:12] = input_assignments(12)
    actual, phases, _ = propagate(exported, initial=physical_initial)
    np.testing.assert_array_equal(actual[:11], physical_initial[:11])
    np.testing.assert_array_equal(
        actual[mapping[11]], physical_initial[11] ^ np.tile(EXPECTED, 2)
    )
    assert not actual[12:].any()
    np.testing.assert_allclose(phases, 1, atol=ATOL, rtol=0)
    return circuit, mapping


CIRCUITS = {}
MAPPINGS = {}
resources = []
for name, records in VARIANTS.items():
    circuit, mapping = compact_circuit(records, name)
    CIRCUITS[name], MAPPINGS[name] = circuit, mapping
    counts = Counter((g["kind"], len(g["controls"])) for g in records)
    resources.append(
        {
            "circuit": name,
            "allocated_qubits": circuit.num_qubits,
            "dedicated_scratch": circuit.num_qubits - 12,
            "operations": len(records),
            "X": counts["x", 0],
            "CX": counts["x", 1],
            "CCX": counts["x", 2],
            "C3X": counts["x", 3],
            "CCRx_pi": counts["rx", 2],
        }
    )
    with (OUT / f"{name}.qpy").open("wb") as handle:
        qpy.dump(circuit, handle)
assert CIRCUITS["borrow_q9"].num_qubits == 18
assert CIRCUITS["borrow_q4_q9"].num_qubits == 17
assert CIRCUITS["borrow_q12_q4_q9"].num_qubits == 16
assert CIRCUITS["clean_reuse_q12"].num_qubits == 18
display(pd.DataFrame(resources))
display(
    pd.DataFrame(
        [
            {"original_wire": old, "physical_wire": new}
            for old, new in MAPPINGS["borrow_q4_q9"].items()
        ]
    )
)

## 8. Qiskit cross-check on an entangled complex state

The exhaustive monomial proof above covers all allowed states. This separate
statevector calculation checks Qiskit's actual exported circuits with the input
and output registers entangled with an untouched external reference. It tests
both the XOR oracle and the phase wrapper with complex amplitudes and fixed
phase conventions. No measurements or sampled counts are used.

In [ ]:
rng = np.random.default_rng(20260920)
logical_amplitudes = rng.normal(size=(2, 4096)) + 1j * rng.normal(size=(2, 4096))
logical_amplitudes /= np.linalg.norm(logical_amplitudes)
phase_amplitudes = logical_amplitudes[:, :2048].copy()
phase_amplitudes /= np.linalg.norm(phase_amplitudes)
statevector_checks = []
for name, compute in CIRCUITS.items():
    width = compute.num_qubits
    for mode in ("xor", "phase"):
        initial = np.zeros(1 << (width + 1), dtype=complex)
        expected = np.zeros_like(initial)
        amplitudes = logical_amplitudes if mode == "xor" else phase_amplitudes
        for ref in (0, 1):
            indices = np.arange(amplitudes.shape[1]) + (ref << width)
            initial[indices] = amplitudes[ref]
            if mode == "xor":
                destinations = indices ^ (np.tile(EXPECTED, 2).astype(int) << 11)
                expected[destinations] = amplitudes[ref]
            else:
                expected[indices] = amplitudes[ref] * (1 - 2 * EXPECTED.astype(int))
        circuit = compute.copy()
        if mode == "phase":
            circuit.z(11)
            circuit.compose(compute.inverse(), inplace=True)
        actual = Statevector(initial).evolve(circuit, qargs=range(width)).data
        error = float(np.max(np.abs(actual - expected)))
        np.testing.assert_allclose(actual, expected, atol=ATOL, rtol=0)
        statevector_checks.append(
            {"circuit": name, "mode": mode, "max_amplitude_error": error}
        )
display(pd.DataFrame(statevector_checks))

## 9. Full circuits with annotated Boolean stages

Each circuit stays on one canvas, in the same style as the guarded phase-oracle
study. **Vertical dashed lines** delimit A–G, the output consumer, exact inverse
cleanup, and the final X. The clean-reuse comparison also shows its separate
**Clear A** stage. Every displayed wire has an **in** and **out** value at every
stage. Idle inputs q0 and q1 are hidden from the drawings only; remaining wires
keep their original labels. The exported circuits retain the complete input
interface, and resource counts still include both hidden inputs.
The six drawings include the CCX reference, all four conditional-borrowing
variants, and the globally clean q12 alternative.

Formulas above the gates define the Boolean values; notes below describe guards
and restoration order. Blue outgoing values mark changed expressions. The
output row is yellow, conditionally borrowed rows are green, and globally reused
q12 is pale blue. Boundary labels are **global expressions**: in particular,
$r_{12}=A\oplus C$, $r_4=(x_4\oplus x_8)\oplus E$, and $r_9=x_9\oplus G$.
The identities $B r_{12}=C$, $D r_4=E$, and $F r_9=G$ explain the added controls.

The input symbols $x_i$ and initial target $y$ label computational-basis branches.
The consumer first produces $y\oplus h$, where $h=\overline{x_3}G$.
Cleanup preserves that output; the final X gives $y\oplus f$ with $f=\overline h$.
These are the complete XOR computations; the phase wrapper subsequently applies
Z to q11 and reverses the complete computation.

Gate symbols, open controls, and three-control gates are read from the actual
Qiskit exports. Every incoming and outgoing boundary label is independently
checked on all **4,096 input-and-target basis columns**. The rotation baseline
is represented here by its CCX abstraction; its phases are checked in sections
6 and 8. PNG and SVG figures are exported to `annotated_circuits/`, with copies
at the existing `*_circuit` paths. `circuit_annotations.json` records the checked
stage intervals and labels.

In [ ]:
from matplotlib.patches import Circle
from IPython.display import Image as NotebookImage


def annotation_values() -> dict[str, np.ndarray]:
    """Evaluate the displayed expressions for every input and arbitrary target y."""
    logical = input_assignments(12)
    values = {f"x_{{{q}}}": logical[q] for q in range(11)}
    values.update({key: np.tile(value, 2) for key, value in SYMBOLS.items()})
    values.update(
        {
            "0": np.zeros(4096, dtype=bool),
            "y": logical[11],
            "u": logical[4] ^ logical[8],
            "r_{12}": np.tile(A ^ C, 2),
            "r_4": np.tile(u ^ E, 2),
            "r_9": np.tile(INPUTS[9] ^ G, 2),
            r"y\oplus h": logical[11] ^ np.tile(~EXPECTED, 2),
            r"y\oplus f": logical[11] ^ np.tile(EXPECTED, 2),
        }
    )
    return values


def annotated_stages(name: str) -> tuple[list[dict], set[int]]:
    """Specify exact instruction intervals and global values at their boundaries."""
    borrow_12 = name in ("borrow_q12", "borrow_q12_q4_q9")
    borrow_4 = name in ("borrow_q4_q9", "borrow_q12_q4_q9")
    borrow_9 = name in ("borrow_q9", "borrow_q4_q9", "borrow_q12_q4_q9")
    clean_12 = name == "clean_reuse_q12"
    borrowed = {
        q for q, used in ((12, borrow_12), (4, borrow_4), (9, borrow_9)) if used
    }
    stages = []

    def add(title, count, formula, updates, note, identity="", width=5.2):
        start = stages[-1]["stop"] if stages else 0
        stages.append(
            dict(
                title=title,
                start=start,
                stop=start + count,
                formula=formula,
                updates=updates,
                note=note,
                identity=identity,
                width=width,
            )
        )

    add("A", 1, r"A=x_2x_8", {12: "A"}, r"$q_{12}:\ 0\to A$")
    add(
        "B",
        3,
        r"B=\overline{x_8\oplus x_7}\,\overline{A}",
        {13: "B"},
        "$q_8$ is restored\n$B$ is preserved while C is used",
        r"B=1\Rightarrow A=0",
        width=6.7,
    )
    if clean_12:
        add(
            "Clear A",
            1,
            r"q_{12}:\ A\oplus A=0",
            {12: "0"},
            "Globally clean reuse\nB remains stored on $q_{13}$",
            width=6.0,
        )
    c_wire = 12 if borrow_12 or clean_12 else 14
    add(
        "C",
        3,
        r"C=\overline{x_8\oplus x_5}\,B",
        {c_wire: "r_{12}" if borrow_12 else "C"},
        (
            r"$B=1\Rightarrow q_{12}=0$ before C" + "\n" + r"$B r_{12}=C$; retain B"
            if borrow_12
            else f"$q_{{{c_wire}}}:\\ 0\\to C$\n$q_8$ is restored"
        ),
        r"r_{12}=A\oplus C" if borrow_12 else "",
        width=6.7,
    )
    add(
        "D",
        4 if borrow_4 else 5,
        r"D=\overline{x_4\oplus x_8}\,\overline{x_8\oplus x_6}",
        {15: "D", 4: "u" if borrow_4 else "x_{4}"},
        (
            r"$D=1\Rightarrow u=0$" + "\nDelay restoration of $q_4$"
            if borrow_4
            else "$q_4, q_8$ are restored\nD is stored on $q_{15}$"
        ),
        r"u=x_4\oplus x_8",
        width=8.3,
    )
    add(
        "E",
        1,
        r"E=CD",
        {4 if borrow_4 else 16: "r_4" if borrow_4 else "E"},
        (r"$D r_4=E$; retain D" if borrow_4 else r"$q_{16}:\ 0\to E$")
        + ("\n" + r"$E=B r_{12}D$; retain B" if borrow_12 else ""),
        r"r_4=u\oplus E" if borrow_4 else (r"E=B r_{12}D" if borrow_12 else ""),
        width=6.0,
    )
    add(
        "F",
        1,
        r"F=\overline{x_9}E",
        {17: "F"},
        "Globally correct F on $q_{17}$\n" + r"$F=1\Rightarrow x_9=0$",
        r"F=\overline{x_9}D r_4" if borrow_4 else "",
        width=6.0,
    )
    add(
        "G",
        1,
        r"G=\overline{x_{10}}F",
        {9 if borrow_9 else 18: "r_9" if borrow_9 else "G"},
        (
            r"$F=1\Rightarrow q_9=0$ before G" + "\n" + r"$F r_9=G$; retain F"
            if borrow_9
            else r"$q_{18}:\ 0\to G$" + "\nDedicated scratch"
        ),
        r"r_9=x_9\oplus G" if borrow_9 else "",
        width=6.0,
    )
    prefix_length = stages[-1]["stop"]
    add(
        "Consume",
        1,
        r"h=\overline{x_3}G",
        {11: r"y\oplus h"},
        "Toggle output $q_{11}$\nOuter complement is still pending",
        r"h=\overline{x_3}F r_9" if borrow_9 else "",
        width=6.4,
    )
    restored = {q: f"x_{{{q}}}" if q < 11 else "0" for q in MAPPINGS[name] if q != 11}
    cleanup_notes = ["Undo the exact prefix in reverse order; output is preserved."]
    if borrow_9:
        cleanup_notes.append(r"Restore $q_9$ before erasing its protecting flag F.")
    if borrow_4:
        cleanup_notes.append(
            r"Erase E from $q_4$ before erasing D; finally restore $x_4$."
        )
    if borrow_12:
        cleanup_notes.append(r"Restore A in $q_{12}$ before erasing B, then erase A.")
    if clean_12:
        cleanup_notes.append(r"Erase C, recompute A before erasing B, then erase A.")
    if not borrowed and not clean_12:
        cleanup_notes.append("Erase G, F, E, D, C, B, A; restore all temporary XORs.")
    add(
        "Uncompute",
        prefix_length,
        r"U_{\mathrm{prefix}}^{-1}",
        restored,
        "\n".join(cleanup_notes),
        r"q_i=x_i\ (i<11),\quad q_{12\ldots}=0",
        width=3.8 + 0.82 * prefix_length,
    )
    add(
        "Complement",
        1,
        r"f=\overline{h}",
        {11: r"y\oplus f"},
        r"$q_{11}:\ y\oplus h\to y\oplus f$" + "\nFinal X is included",
        width=6.6,
    )
    return stages, borrowed


def draw_annotated_circuit(name: str) -> dict:
    """Draw and exhaustively check every boundary of the actual exported circuit."""
    compute, mapping = CIRCUITS[name], MAPPINGS[name]
    # Read gates back from the Qiskit circuit, then restore original wire labels.
    inverse_mapping = {new: old for old, new in mapping.items()}
    records = _qiskit_records(compute)
    for record in records:
        record["controls"] = [inverse_mapping[q] for q in record["controls"]]
        record["target"] = inverse_mapping[record["target"]]
        assert record["kind"] == "x"
    hidden_idle_wires = {0, 1}
    used_wires = {
        q for record in records for q in [*record["controls"], record["target"]]
    }
    assert hidden_idle_wires.isdisjoint(used_wires)
    displayed_wires = [q for q in mapping if q not in hidden_idle_wires]
    _, _, trace = propagate(records, logical_bits=12)
    stages, borrowed = annotated_stages(name)
    assert stages[-1]["stop"] == len(records)
    values = annotation_values()
    symbols = {q: f"x_{{{q}}}" if q < 11 else "y" if q == 11 else "0" for q in mapping}
    edges = np.concatenate(([0.0], np.cumsum([stage["width"] for stage in stages])))
    width = compute.num_qubits
    figure, ax = plt.subplots(figsize=(44, 17), facecolor="white")
    ax.set_position([0.028, 0.095, 0.955, 0.80])
    ax.set_xlim(-3.0, edges[-1] + 0.3)
    ax.set_ylim(-len(displayed_wires) - 4.0, 4.4)
    ax.set_aspect("equal")
    ax.set_axis_off()
    gate_color, changed_color, conditional_color = "#193c64", "#175d9c", "#197052"
    wire_positions = {q: -row for row, q in enumerate(displayed_wires)}

    for q, y_position in wire_positions.items():
        if q == 11 or q in borrowed or (name == "clean_reuse_q12" and q == 12):
            color = "#fff4cf" if q == 11 else "#e8f4ed" if q in borrowed else "#e8eef9"
            ax.axhspan(y_position - 0.36, y_position + 0.36, color=color, zorder=0)
        ax.plot([-0.7, edges[-1]], [y_position] * 2, color="#a8b1bd", lw=0.85, zorder=1)
        ax.text(-1.9, y_position, rf"$q_{{{q}}}$", ha="right", va="center", fontsize=13)
        ax.text(
            -0.85,
            y_position,
            f"$|{symbols[q]}\\rangle$",
            ha="right",
            va="center",
            fontsize=11,
        )
    ax.text(-1.9, 0.75, "Qubit", ha="right", fontsize=10, color="#52606d")
    for boundary in edges:
        ax.plot(
            [boundary] * 2,
            [-len(displayed_wires) - 3.6, 4.0],
            color="#8793a2",
            lw=1,
            linestyle=(0, (5, 5)),
            zorder=1,
        )

    checked_stages = []
    for index, stage in enumerate(stages):
        left, right = edges[index : index + 2]
        middle = (left + right) / 2
        before = symbols.copy()
        symbols.update(stage["updates"])
        for q in mapping:
            np.testing.assert_array_equal(trace[stage["start"]][q], values[before[q]])
            np.testing.assert_array_equal(trace[stage["stop"]][q], values[symbols[q]])
        for q, y_position in wire_positions.items():
            for position, value, changed in (
                (left + 0.7, before[q], False),
                (right - 0.7, symbols[q], before[q] != symbols[q]),
            ):
                ax.text(
                    position,
                    y_position + 0.12,
                    f"${value}$",
                    ha="center",
                    va="bottom",
                    fontsize=11,
                    color=changed_color if changed else "#4b5664",
                    bbox={"facecolor": "white", "edgecolor": "none", "pad": 0.3},
                    zorder=6,
                )
        ax.text(
            middle,
            3.4,
            stage["title"],
            ha="center",
            fontsize=16,
            fontweight="bold",
            color=gate_color,
        )
        ax.text(middle, 2.55, f"${stage['formula']}$", ha="center", fontsize=13)
        if stage["identity"]:
            ax.text(
                middle,
                1.65,
                f"${stage['identity']}$",
                ha="center",
                fontsize=12,
                color=conditional_color,
            )
        for position, label in ((left + 0.7, "in"), (right - 0.7, "out")):
            ax.text(position, 0.75, label, ha="center", fontsize=10, color="#52606d")
        operations = records[stage["start"] : stage["stop"]]
        locations = np.linspace(left + 1.65, right - 1.65, len(operations))
        if len(operations) == 1:
            locations[:] = middle
        for position, record in zip(locations, operations):
            qubits = [*record["controls"], record["target"]]
            ys = [wire_positions[q] for q in qubits]
            if record["controls"]:
                ax.plot(
                    [position] * 2,
                    [min(ys), max(ys)],
                    color=gate_color,
                    lw=1.4,
                    zorder=2,
                )
                for q, polarity in zip(record["controls"], record["polarity"]):
                    ax.add_patch(
                        Circle(
                            (position, wire_positions[q]),
                            0.08,
                            facecolor=gate_color if polarity else "white",
                            edgecolor=gate_color,
                            lw=1.4,
                            zorder=4,
                        )
                    )
            target_y = wire_positions[record["target"]]
            ax.add_patch(
                Circle(
                    (position, target_y),
                    0.16,
                    facecolor="white",
                    edgecolor=gate_color,
                    lw=1.4,
                    zorder=4,
                )
            )
            ax.plot(
                [position - 0.11, position + 0.11],
                [target_y] * 2,
                color=gate_color,
                lw=1.1,
                zorder=5,
            )
            ax.plot(
                [position] * 2,
                [target_y - 0.11, target_y + 0.11],
                color=gate_color,
                lw=1.1,
                zorder=5,
            )
        ax.text(
            middle,
            -len(displayed_wires) - 0.2,
            stage["note"],
            ha="center",
            va="top",
            fontsize=11,
            linespacing=1.7,
            color=conditional_color if borrowed else gate_color,
        )
        checked_stages.append(
            {
                "stage": stage["title"],
                "start": stage["start"],
                "stop": stage["stop"],
                "in": before,
                "out": symbols.copy(),
            }
        )

    titles = {
        "ccx_reference": "Original reference — CCX abstraction",
        "borrow_q9": "Borrow q9 under F",
        "borrow_q4_q9": "Borrow q4 and q9 under D and F",
        "borrow_q12": "Borrow q12 under B",
        "borrow_q12_q4_q9": "Borrow q12, q4 and q9 under B, D and F",
        "clean_reuse_q12": "Reuse q12 after globally uncomputing A",
    }
    figure.text(
        0.03, 0.96, titles[name], fontsize=24, fontweight="bold", color=gate_color
    )
    figure.text(
        0.03,
        0.923,
        f"int2float output 3  •  {width} allocated qubits ({len(displayed_wires)} shown)  •  {len(records)} gates  •  "
        "Full XOR computation, inverse cleanup and final complement",
        fontsize=14,
        color="#52606d",
    )
    figure.text(
        0.03,
        0.052,
        "Dashed lines delimit stages; every wire shows global in / out values. "
        "Blue values change; yellow marks the output; green marks conditional borrowing; pale blue marks globally clean reuse.",
        fontsize=12,
        color="#52606d",
    )
    figure.text(
        0.03,
        0.027,
        "Original wire labels are retained. Idle q0 and q1 are hidden; allocation counts and exports still include them. "
        "xᵢ and y label computational-basis branches. All boundaries are checked on 4,096 input-and-target columns.",
        fontsize=12,
        color="#52606d",
    )
    directory = OUT / "annotated_circuits"
    directory.mkdir(exist_ok=True)
    exports = []
    for extension in ("png", "svg"):
        destination = directory / f"{name}_annotated.{extension}"
        figure.savefig(destination, dpi=170, bbox_inches="tight", facecolor="white")
        # Keep the study's established figure paths useful in newly generated runs.
        (OUT / f"{name}_circuit.{extension}").write_bytes(destination.read_bytes())
        exports.append(str(destination.relative_to(OUT)))
    display(NotebookImage(filename=str(directory / f"{name}_annotated.png")))
    plt.close(figure)
    return {
        "circuit": name,
        "basis_columns_checked": 4096,
        "allocated_qubits": width,
        "displayed_wires": displayed_wires,
        "hidden_idle_wires": sorted(hidden_idle_wires),
        "rendered_operations": len(records),
        "borrowed_wires": sorted(borrowed),
        "stages": checked_stages,
        "exports": exports,
    }


annotation_checks = [
    draw_annotated_circuit(name) for name in VARIANTS if name != "original_rotations"
]
(OUT / "circuit_annotations.json").write_text(
    json.dumps(annotation_checks, indent=2) + "\n"
)
print(
    "PASS: all circuit boundary labels agree with the exported gates on 4,096 basis columns."
)

## 10. What is established, and what the next algorithm would need

| Construction | Qubits | Scratch | X | CX | CCX | 3-controlled X |
|---|---:|---:|---:|---:|---:|---:|
| CCX abstraction of saved reference | 19 | 7 | 1 | 16 | 15 | 0 |
| Borrow q9 under F | 18 | 6 | 1 | 16 | 14 | 1 |
| Borrow q4 under D, then q9 under F | 17 | 5 | 1 | 14 | 12 | 3 |
| Borrow q12 under B | 18 | 6 | 1 | 16 | 13 | 2 |
| Borrow q12, q4, and q9 under B, D, F | 16 | 4 | 1 | 14 | 10 | 5 |
| Clean q12 by uncomputing A before C | 18 | 6 | 1 | 16 | 17 | 0 |

These counts describe the complete **XOR computation including its own cleanup**.
A compute–Z–uncompute phase wrapper doubles them and adds one Z, keeping the same
width. They are logical gate counts, not T counts or post-decomposition costs.
The added multi-control consumers may cost more to implement; decomposition may
also require ancillas. No claim is made that this is the best phase-only construction.

The useful rule is not simply “a wire becomes conditionally clean.” It is:

1. Establish a nonempty predicate P and a known current wire value on P.
2. Preserve the predicate while borrowing the wire; normalize a known 1 if needed.
3. Redirect a target only where the Boolean data dependencies allow it.
4. Ensure every consumer is insensitive to the borrowed wire's residual value on
   P=0. Here that required explicit extra controls.
5. Delay restoration when necessary, then undo the borrowing before erasing its
   protecting flag or reusing the input in its original role.
6. Recheck the whole output function, phases, and restoration, then account for
   the guard and cleanup cost.

The three conditional relocations and the clean-reuse comparison form concrete
next test cases for a synthesis pass. Candidates can include existing workspace
wires such as q12, not just original system inputs.
The single-flag inventory and composite examples are candidate discovery tools;
they do not automatically certify arbitrary target moves or arbitrary predicates.

Input provenance is stored with the dataset snapshot. Background on the resource:
[Khattar and Gidney, *Rise of conditionally clean ancillae*](https://quantum-journal.org/papers/q-2025-05-21-1752/).

In [ ]:
report = {
    "source_sha256": hashlib.sha256(SOURCE.read_bytes()).hexdigest(),
    "selected_output": 3,
    "provenance": DATA["provenance"],
    "conditional_system_facts": facts,
    "composite_facts": composite_facts,
    "rejected_attempts": failures,
    "verification": verification,
    "statevector_checks": statevector_checks,
    "resources": resources,
    "original_to_physical": MAPPINGS,
}
(OUT / "verification.json").write_text(json.dumps(report, indent=2) + "\n")
(OUT / "transformed_records.json").write_text(json.dumps(VARIANTS, indent=2) + "\n")
pd.DataFrame(resources).to_csv(OUT / "resources.csv", index=False)
print("All checks passed; results and full circuit figures saved to", OUT)